# 06 — Advanced Task: Transfer Learning (ACC Arena → Salt & Tar)

**Project #8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**  
Network Data Analysis Laboratory 2025-2026, Politecnico di Milano

---

## Goal (3 points)

> **Transfer Learning from ACC Arena to Salt & Tar scenario.**  
> Compare with the same model trained on a **limited Salt&Tar training set**.

### Experimental design

| Strategy | Description |
|----------|-------------|
| **Fine-tune (TL)** | Load MLP weights trained on full ACC Arena, freeze early layers, fine-tune top layers on a limited Salt&Tar training set |
| **Scratch (baseline)** | Train the same MLP architecture from random initialisation on the same limited Salt&Tar training set |

We compare the two strategies as a function of the **Salt&Tar training set size** to show the benefit of transfer learning when labelled data is scarce.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import json
from pathlib import Path

from src.models.neural_network import MLPRegressor, predict_mlp, DEVICE
from src.models.transfer_learning import TransferLearner
from src.utils.metrics import evaluate, print_metrics
from src.utils.visualization import (
    plot_training_history,
    plot_pred_vs_true,
    plot_transfer_learning_comparison,
)

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../results/models')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SAVE_FIGS = True

print(f'Device: {DEVICE}')

## 1. Load Salt & Tar data

In [ ]:
X_salt_train_full = np.load(PROCESSED_DIR / 'salt_X_train.npy')
X_salt_val        = np.load(PROCESSED_DIR / 'salt_X_val.npy')
X_salt_test       = np.load(PROCESSED_DIR / 'salt_X_test.npy')
y_salt_train_full = np.load(PROCESSED_DIR / 'salt_y_train.npy')
y_salt_val        = np.load(PROCESSED_DIR / 'salt_y_val.npy')
y_salt_test       = np.load(PROCESSED_DIR / 'salt_y_test.npy')

print(f"Salt & Tar — train: {X_salt_train_full.shape}, val: {X_salt_val.shape}, test: {X_salt_test.shape}")

INPUT_DIM = X_salt_test.shape[1]
print(f"Feature dimension: {INPUT_DIM}")

## 2. Load best ACC Arena model

In [ ]:
with open(MODELS_DIR / 'training_results.json') as f:
    training_results = json.load(f)

best_nn = training_results['best_nn_params']
HIDDEN_DIMS = [best_nn[f'h{i}'] for i in range(best_nn['n_layers'])]
DROPOUT     = best_nn['dropout']

BASE_MODEL_PATH = MODELS_DIR / 'mlp_acc_arena_tuned_base.pt'
print(f"Base model: {BASE_MODEL_PATH}")
print(f"Architecture: {INPUT_DIM} → {HIDDEN_DIMS} → 1")

## 3. Full Salt&Tar training set — upper bound

In [ ]:
# Upper bound: train from scratch on full Salt&Tar data
model_full, hist_full = TransferLearner.train_from_scratch(
    X_salt_train_full, y_salt_train_full,
    X_salt_val, y_salt_val,
    input_dim=INPUT_DIM,
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
    lr=best_nn['lr'],
    max_epochs=100, patience=15,
    checkpoint_path=MODELS_DIR / 'mlp_salt_tar_scratch_full.pt',
)

preds_full = predict_mlp(model_full, X_salt_test)
metrics_full_scratch = evaluate(y_salt_test, preds_full, hist_full['training_time_s'])
print_metrics(metrics_full_scratch, 'Scratch (full Salt&Tar train set) — UPPER BOUND')

## 4. Experiment: vary Salt&Tar training set size

In [ ]:
# Fractions of the full Salt&Tar training set to use
TRAIN_FRACTIONS = [0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00]
TRAIN_SIZES = [max(32, int(len(X_salt_train_full) * f)) for f in TRAIN_FRACTIONS]

print('Training set sizes to evaluate:')
print(TRAIN_SIZES)

In [ ]:
results_finetune = []   # metrics for each train size — fine-tuned model
results_scratch  = []   # metrics for each train size — scratch model

for n_train in TRAIN_SIZES:
    # Subsample training set
    idx = np.random.choice(len(X_salt_train_full), n_train, replace=False)
    X_tr_sub = X_salt_train_full[idx]
    y_tr_sub = y_salt_train_full[idx]

    print(f"\n{'─'*55}")
    print(f" Training size: {n_train} ({100*n_train/len(X_salt_train_full):.1f}%)")
    print('─'*55)

    # --- Strategy 1: Fine-tuning ---
    tl = TransferLearner(
        base_model_path=BASE_MODEL_PATH,
        input_dim=INPUT_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    )
    hist_ft = tl.fine_tune(
        X_tr_sub, y_tr_sub, X_salt_val, y_salt_val,
        n_layers_to_keep_trainable=1,
        lr=1e-4,
        max_epochs=50, patience=10,
        checkpoint_path=MODELS_DIR / f'mlp_salt_finetune_{n_train}.pt',
    )
    preds_ft = tl.predict(X_salt_test)
    m_ft = evaluate(y_salt_test, preds_ft, hist_ft['training_time_s'])
    results_finetune.append(m_ft)
    print_metrics(m_ft, f'Fine-tune   n={n_train}')

    # --- Strategy 2: From scratch ---
    model_sc, hist_sc = TransferLearner.train_from_scratch(
        X_tr_sub, y_tr_sub, X_salt_val, y_salt_val,
        input_dim=INPUT_DIM,
        hidden_dims=HIDDEN_DIMS, dropout=DROPOUT,
        lr=best_nn['lr'],
        max_epochs=100, patience=15,
        checkpoint_path=MODELS_DIR / f'mlp_salt_scratch_{n_train}.pt',
    )
    preds_sc = predict_mlp(model_sc, X_salt_test)
    m_sc = evaluate(y_salt_test, preds_sc, hist_sc['training_time_s'])
    results_scratch.append(m_sc)
    print_metrics(m_sc, f'Scratch     n={n_train}')

print('\nAll TL experiments complete.')

## 5. Visualise results

In [ ]:
# RMSE vs. training set size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, ylabel in [
    (axes[0], 'rmse',  'RMSE (Mbps)'),
    (axes[1], 'mae',   'MAE  (Mbps)'),
    (axes[2], 'r2',    'R²'),
]:
    ft_vals = [m[metric] for m in results_finetune]
    sc_vals = [m[metric] for m in results_scratch]

    ax.plot(TRAIN_SIZES, ft_vals, marker='o', label='Fine-tune (TL)', color='steelblue')
    ax.plot(TRAIN_SIZES, sc_vals, marker='s', label='Scratch',        color='coral')

    # Full-data scratch upper bound
    ax.axhline(metrics_full_scratch[metric], linestyle='--', color='gray',
               alpha=0.7, label='Scratch (full data)')

    ax.set_xlabel('Salt&Tar training set size')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend()
    ax.set_xscale('log')

plt.suptitle('Transfer Learning: Fine-tune vs. Scratch — Salt & Tar', fontsize=14)
plt.tight_layout()
if SAVE_FIGS:
    from pathlib import Path
    Path('../results/figures').mkdir(exist_ok=True)
    plt.savefig('../results/figures/06_tl_metric_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Results Table

In [ ]:
rows = []
for n, m_ft, m_sc in zip(TRAIN_SIZES, results_finetune, results_scratch):
    rows.append({'n_train': n,
                 'finetune_rmse': m_ft['rmse'], 'finetune_mae': m_ft['mae'], 'finetune_r2': m_ft['r2'],
                 'scratch_rmse':  m_sc['rmse'], 'scratch_mae':  m_sc['mae'], 'scratch_r2':  m_sc['r2']})

tl_df = pd.DataFrame(rows).set_index('n_train').round(4)
print('=== Transfer Learning Results ===')
display(tl_df)
print('\nLatex:')
print(tl_df.to_latex(float_format='%.4f'))

## 7. Best configuration: detailed evaluation

In [ ]:
# Use the best TL model (lowest RMSE fine-tune)
best_n = TRAIN_SIZES[np.argmin([m['rmse'] for m in results_finetune])]
print(f'Best fine-tune training size: {best_n}')

best_tl = TransferLearner(BASE_MODEL_PATH, INPUT_DIM, HIDDEN_DIMS, DROPOUT)
best_tl.model = MLPRegressor(INPUT_DIM, HIDDEN_DIMS, DROPOUT)
best_tl.model.load_state_dict(
    torch.load(MODELS_DIR / f'mlp_salt_finetune_{best_n}.pt', map_location=DEVICE)
)
preds_best_tl = best_tl.predict(X_salt_test)

fig = plot_pred_vs_true(y_salt_test, preds_best_tl,
                        model_name=f'TL Fine-tune n={best_n}', save=SAVE_FIGS)
plt.show()

## 8. Key Findings

> Fill in after running:

- **Transfer Learning advantage**: With only …% of Salt&Tar training data, fine-tuning achieves RMSE of … vs. … for scratch training.
- **Break-even point**: Fine-tuning and scratch become comparable at approximately … training samples.
- **Best fine-tune configuration**: n_layers_to_keep_trainable=1, lr=1e-4
- **Conclusion**: Transfer learning is most beneficial when labelled data is …